# Render Free Recall Fitting

Orchestrator: dispatches `templates/free_recall_fitting.ipynb` via papermill for each model configuration in the factorial comparison.

In [ ]:
import papermill as pm
from cru_to_cmr.helpers import find_project_root
from cru_to_cmr.config import base_model_configs, compterm_model_configs, BASE_PARAMS

ROOT = find_project_root()

In [ ]:
# Shared parameters
shared_params = {
    "data_name": "HealeyKahana2014",
    "data_query": "data['listtype'] == -1",
    "data_path": f"{ROOT}/data/HealeyKahana2014.h5",
    "base_params": BASE_PARAMS,
    "redo_fits": False,
    "redo_sims": True,
    "run_tag": "Fitting",
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
    "best_of": 3,
    "experiment_count": 50,
    "seed": 0,
    "analysis_paths": [
        "cru_to_cmr.analyses.spc.plot_spc",
        "cru_to_cmr.analyses.crp.plot_crp",
        "cru_to_cmr.analyses.pnr.plot_pnr",
    ],
}

# Optional: set to a model name to run only that config, or None for all
target_model = None

## Dispatch all configs

In [ ]:
template = f"{ROOT}/analyses/templates/free_recall_fitting.ipynb"
data_name = shared_params["data_name"]
run_tag = shared_params["run_tag"]
rendered_dir = f"{ROOT}/analyses/rendered"

configs = [
    ("base", base_model_configs),
    ("compterm", compterm_model_configs),
]

for factory_type, model_configs in configs:
    for model_name, bounds in model_configs.items():
        if target_model is not None and model_name != target_model:
            continue

        file_model_name = model_name.replace(" ", "_")
        output_path = f"{rendered_dir}/fitting_{data_name}_{file_model_name}_{run_tag}.ipynb"

        params = shared_params | {
            "model_name": model_name,
            "factory_type": factory_type,
            "bounds": bounds,
        }

        print(f"Rendering: {model_name} ({factory_type})")
        pm.execute_notebook(
            template,
            output_path,
            parameters=params,
            progress_bar=True,
        )
        print(f"  -> {output_path}")